In [ ]:
import os
from pyspark.sql import SparkSession
ss=SparkSession.builder.appName('app_for_struct_stream').getOrCreate()

In [ ]:
ss

In [ ]:
from pyspark.sql.types import *
# 1. Define the schema in the begining to have read stream work on empty folder
user_schema = StructType([
                            StructField("empid", IntegerType(), True),
                            StructField("login_attempt_time", TimestampType(), True),
                            StructField("success_or_failure", StringType(), True),
                            StructField("file_batch_id", IntegerType(), True)
                        ])

In [ ]:
ss.conf.set("spark.sql.streaming.schemaInference", True)
read_stream_df =    ss.readStream.format("csv")\
                    .options(header = True,delimiter = ",", recursiveFileLookup = True )\
                    .schema(user_schema)\
                    .load("file:///home/yv05/Documents/y_claude_code_folder/y_spark_related/y_env_for_struct_stream/y_data_related_folder/y_input_folder/")

In [ ]:
from pyspark.sql.functions import udf
from datetime import datetime,timedelta
from pyspark.sql.types import *

@udf
def func_date_time():
    return str(datetime.now())

transform_df = read_stream_df.withColumn("Read_time",func_date_time().cast(TimestampType()))

In [ ]:
from pyspark.sql.functions import *
transform_df_change_type = transform_df.withColumn("empid", col("empid").cast(IntegerType()))\
                                        .withColumn("login_attempt_time", col("login_attempt_time").cast(TimestampType()))\
                                        .withColumn("success_or_failure", col("success_or_failure").cast(StringType()))\
                                        .withColumn("file_batch_id", col("file_batch_id").cast(IntegerType()))

transform_df_change_type.printSchema()

In [ ]:
writing_df_append =    transform_df_change_type.writeStream\
                .format("csv")\
                .options(header = True,delimiter = ",")\
                .option("path", "file:///home/yv05/Documents/y_claude_code_folder/y_spark_related/y_env_for_struct_stream/y_data_related_folder/y_output_folder")\
                .option("checkpointLocation","file:///home/yv05/Documents/y_claude_code_folder/y_spark_related/y_env_for_struct_stream/y_data_related_folder/y_checkpointing_location/")\
                .outputMode("append")\
                .start()
writing_df_append.awaitTermination()

In [ ]:
writing_df_append =    transform_df_change_type.withWatermark("my_time_type_tmstmp", "0 seconds")\
                .groupBy(
                    window(col("my_time_type_tmstmp"), "10 seconds"),
                    col("col_company")
                ) \
                .agg(count("*").alias("event_count")) \
                .writeStream\
                .format("parquet")\
                .options(header = True,delimiter = ",")\
                .option("path", "file://"+os.environ['my_home_directory']+"/Documents/y_compendium/y_youtube_teaching/y_pyspark/y_datafiles/y_output_of_ss/")\
                .option("checkpointLocation","file://"+os.environ['my_home_directory']+"/Documents/y_compendium/y_youtube_teaching/y_pyspark/y_datafiles/y_checkpointing_location/")\
                .outputMode("append")\
                .start()
writing_df_append.awaitTermination()